---
title: Keecas Quarto Example
toc: true
format:
    html:
        include-in-header:
            # this is needed for equation numbering and labeling in HTML output             
            - text: | 
                <script>
                MathJax = { tex: { tags: 'ams' } };
                </script>
            # this get rid of unnecessary vertical scrollbars (may need to be adjusted)
            - text: |
                <style>
                .math.display {
                  max-width: 100%;
                  padding-right: 3em;
                }
                </style>
    pdf:
        echo: false
        classoption: [fleqn] # personal preference
echo: true
keep-tex: true
---

{{< pagebreak >}}

# How to use keecas in a Quarto document

This example demonstrates key features of `keecas` for symbolic and units-aware calculations in a Quarto document.

In [1]:
# For quick start use starred import
# from keecas import *

# Or explicit import (this is about all you need)
from keecas import (
    check,                  # main function for verification
    config,                 # configuration options
    pc,                     # pipe commands
    show_eqn,               # main function to generate expression block
    symbols,                # to create symbols (from `sympy`)
    u,                      # unit registry (from pint)
    generate_unique_label,  # label generator
)

In [2]:
# Set the language ('en' is the default)
config.language.language = "it"

::: {.callout-note}

If the rendering engine is `KaTeX` (i.e. Jupyter notebook), the label command `\label{}` will result in a `ParseError`. The latex code will work when rendering by `quarto render`, but since you may want to see the result before rendering the document, an option to disable the `\label{}` command is provided.

When rendering with `quarto` you can have two config:

- if `qmd` files, then everything is fine
- if `ipynb` files, then pass the `--execute` flag to `quarto render` to rerender the notebook

:::


In [3]:
# | eval: false

# This option will not be set at rendering time (only for dev-mode)
config.display.katex = True         # Disable \label command for KaTeX
config.display.print_label = True   # Print labels for easy copy-paste

In [4]:
# Set a prefix for the latex equation (used as namespace)
config.latex.eq_prefix = r"eq-QUARTO_EXAMPLE-"
# config.latex.eq_suffix = ""  # default

# Initialize notebook-global dictionaries for persistence across cells
QUARTO_EXAMPLE = {                      # Main namespace dict for this notebook
    'parameters': (params := {}),       # Global parameters
    'expressions': (eqn := {}),         # Global expressions
    'values': (vals := {}),             # Global evaluated results
}

## Basic Symbolic Math with Units

Define symbols and calculate basic engineering quantities:

In [5]:
# Define symbols with LaTeX notation
F_d, A_load, sigma_Sd = symbols(r"F_{d}, A_{load}, \sigma_{Sd}")

# Cell-local parameters with units
_p = {
    F_d: 93 * u.kN,        # Applied force
    A_load: 50 * u.cm**2,  # Cross-sectional area
}
params.update(_p)  # Save to global params

# Cell-local symbolic expressions
_e = {
    sigma_Sd: "F_d / A_load" | pc.parse_expr,
}
eqn.update(_e)  # Save to global expressions

# Evaluate expressions
_v = {
    k: v | pc.subs(_e | _p) | pc.convert_to([u.MPa]) | pc.N
    for k, v in _e.items()
}

# Descriptions
_d = {
    F_d: "applied force",
    A_load: "cross-sectional area",
    sigma_Sd: "normal stress",
}

# Generate unique labels from keys and descriptions
_l = {
    k: generate_unique_label([k, v]) for k, v in _d.items()
}

show_eqn(
    [_p | _e, _v, _d],
    float_format="{:.2f}",
    label=_l,
    # debug=True,
)

# Labels will be printed for easy reference, only in dev-mode (not rendered by quarto)

F_{d}: eq-QUARTO_EXAMPLE-41g2kkvl
A_{load}: eq-QUARTO_EXAMPLE-3rad1giw
\sigma_{Sd}: eq-QUARTO_EXAMPLE-41gy5jxq


<IPython.core.display.Latex object>

## Pipe Commands Demonstration {#sec-pipe}

Using pipe operators for functional composition:

In [6]:
# Chain operations using pipe commands
x, y, d = symbols("x, y, d")

# Cell-local parameters
_p = {
    x: 3 * u.m,
    y: 4 * u.m,
}

# Symbolic expressions
_e = {
    d: "x^2 + y^2" | pc.parse_expr,
}

# Evaluated results
_v = {
    k: v | pc.subs(_e | _p) | pc.convert_to([u.m]) | pc.N
    for k, v in _e.items()
}

show_eqn([_p | _e, _v])

<IPython.core.display.Latex object>

## Different LaTeX Environments

Different environment are available for display qith `align` being the default. `*` version of the environment are also available.


In [7]:
# Initial setup
E_steel, E_concrete = symbols(r"E_{steel}, E_{concrete}")

_p = {
    E_steel: 200 * u.GPa,
    E_concrete: 30 * u.GPa,
}

_d = {
    E_steel: "steel elastic modulus",
    E_concrete: "concrete elastic modulus",
}

show_eqn([_p, _d])

<IPython.core.display.Latex object>


### Cases Environment

In [8]:
show_eqn([_p, _d], environment="cases")

<IPython.core.display.Latex object>

### Rcases Environment

In [9]:
show_eqn([_p, _d], environment="rcases")

<IPython.core.display.Latex object>

### Equation Environment

In [10]:
from keecas import wrap_column as wc

show_eqn(
    [_p, _d],
    col_wrap=[wc]*2+ [(r'', r'\quad')], # for separating the two expression
    environment="equation"
)

<IPython.core.display.Latex object>

### Gather Environment

### Custom Environment

A custom environment can be defined. Here we define a custom environment with raw LaTeX blocks encapsulation:

In [11]:
a, b, c = symbols(r"a b c")

_p = {
    a: 5 * u.m,
    b: 12 * u.m,
    c: a + b,
}

_d = {
    a: "length a",
    b: "length b",
    c: "total length c",
}

# Custom environment definition with raw LaTeX blocks
custom_env = {
    "separator": " & ",
    "line_separator": r" \\" + "\n",
    "supports_multiple_labels": True,
    "outer_environment": "align",
    "inner_environment": None,
    "inner_prefix": "",
    "inner_suffix": "",
    "outer_prefix": "```{=latex}\n",
    "outer_suffix": "\n```",
    "label_position": "outer",
}

# show_eqn return a Latex object. In this case it is necessary to convert it to Markdown for correct display
from IPython.display import Markdown

Markdown(
    show_eqn(
        [_p, _d],
        environment=custom_env,
        debug=True,
    ).data
)

```{=latex}
\begin{align}
a  &  = 5{\,}\text{m}  &  \quad\text{length a}  \\[8pt]
b  &  = 12{\,}\text{m}  &  \quad\text{length b}  \\[8pt]
c  &  = a + b  &  \quad\text{total length c} 
\end{align}
```


```{=latex}
\begin{align}
a  &  = 5{\,}\text{m}  &  \quad\text{length a}  \\[8pt]
b  &  = 12{\,}\text{m}  &  \quad\text{length b}  \\[8pt]
c  &  = a + b  &  \quad\text{total length c} 
\end{align}
```

In [12]:
# starred version of gather
show_eqn([_p, _d], environment="gather*")

<IPython.core.display.Latex object>

## Equation With Labels {#sec-beam-calc}

Calculate beam deflection with cross-references:

In [13]:
# Beam calculation
q, L, E, I, delta = symbols(r"q, L, E, I, \delta")

_p = {
    q: 5 * u.kN / u.m,
    L: 8 * u.m,
    E: 200 * u.GPa,
    I: 8360 * u.cm**4,
}
params.update(_p)  # Save to global params

# Deflection formula
_e = {
    delta: "5 * q * L^4 / (384 * E * I)" | pc.parse_expr,
}
eqn.update(_e)  # Save to global expressions

_v = {
    k: v | pc.subs(_e | _p) | pc.convert_to([u.mm]) | pc.N
    for k, v in _e.items()
}

_l = {
    k: generate_unique_label([k, v]) for k, v in _e.items()
}

show_eqn([_e, _v], label=_l, float_format="{:.2f}")

\delta: eq-QUARTO_EXAMPLE-2kp5fgk9


<IPython.core.display.Latex object>

::: {.callout-note}

labels emitted by `show_eqn` are latex labels, therefore they need to be referenced with `\eqref{}` or `\ref{}` latex commands:

- `@eq-QUARTO_EXAMPLE-delta` will not work -> (@eq-QUARTO_EXAMPLE-delta)
- `\eqref{eq-QUARTO_EXAMPLE-2kp5fgk9}` will work -> \eqref{eq-QUARTO_EXAMPLE-2kp5fgk9}

:::

The deflection calculated in \eqref{eq-QUARTO_EXAMPLE-2kp5fgk9} shows acceptable values for serviceability.

### Label Generation

Various methods for generating labels for equation cross-references:

In [14]:
# Setup for label examples
from keecas import generate_label

J, E, a, b, c_0 = symbols(r"J, E, a, b, c_{0}")

# Dictionary to be displayed by show_eqn
_p = {
    J: 123 * u.cm**4,
    E: 456 * u.MPa,
    a: "b + c_0 / 2" | pc.parse_expr,
}

#### Manual Label Assignment

When manually assigning labels, they should be LaTeX-safe:

In [15]:
# When manually assigning labels, they should be LaTeX safe
_l = {
    J: "eq-moment-of-inertia",
    E: "eq-modulus-of-elasticity",
    a: "eq-an-expression",
}

# Pass _l to show_eqn
show_eqn(_p, label=_l)

J: eq-moment-of-inertia
E: eq-modulus-of-elasticity
a: eq-an-expression


<IPython.core.display.Latex object>

#### Controlled Automatic Generation

Generate labels with `eq-` prefix automatically:

##### Non hashed Labels

In [16]:
# Provide LaTeX-safe descriptions (no spaces)
_l = {
    J: "moment-of-inertia",
    E: "modulus-of-elasticity",
    a: "an-expression",
}

# Generate the labels with eq- prefix
_l = generate_label(_l)

# Pass _l to show_eqn
show_eqn(_p, label=_l)

J: eq-QUARTO_EXAMPLE-moment-of-inertia
E: eq-QUARTO_EXAMPLE-modulus-of-elasticity
a: eq-QUARTO_EXAMPLE-an-expression


<IPython.core.display.Latex object>

##### hashed Labels (recommended)

In [17]:
# Descriptions with spaces (will be converted to LaTeX-safe format)
_d = {
    J: "moment of inertia",
    E: "modulus of elasticity",
    a: "an expression",
}

# Generate unique labels from descriptions
_l = generate_unique_label(_d)

# save to global dict for later retrieval
labels = _l

# Pass _l to show_eqn
show_eqn(_p, label=_l)

J: eq-QUARTO_EXAMPLE-534vb10v
E: eq-QUARTO_EXAMPLE-1sipf20t
a: eq-QUARTO_EXAMPLE-30phqdq8


<IPython.core.display.Latex object>

With an helper function, the generated labels saved to a dict can be easily referenced in the quarto document:

In [18]:
from IPython.display import Markdown

# helper function for eqref
def eqref(label):
    return Markdown(rf"\eqref{{{label}}}")

For example $J$ is defined at `{python} eqref(labels[J])`, while $E$ is defined at `{python} eqref(labels[E])`

#### Full Automatic Generation

A callable can be passed to `label` argument of `show_eqn`. For each key, all the key and values will be passed to the callable.

In [19]:
# Pass generate_unique_label directly to show_eqn
# Labels are generated from key + values from all the dict passed to show_eqn 

# this is equivalent 
_l = {
    k: generate_unique_label([k, _p[k], _d[k]]) for k in _p.keys()
}

# to this
show_eqn([_p, _d], label=generate_unique_label)

J: eq-QUARTO_EXAMPLE-3lb6w5uv
E: eq-QUARTO_EXAMPLE-1mp6ez2n
a: eq-QUARTO_EXAMPLE-5kh184ai


<IPython.core.display.Latex object>

::: {.callout-note}

When using this method, the interpolated label will change if any of the values in the key-row of the Dataframe change. For a more stable approach, either pass an already interpolated dict of labels as string (recommended), or use a custom callable with a more resilient logic (e.g. use only the element of the list that are less likely to change).

:::

## Check Function

Using the `check` function for design checks:

In [20]:
# Design checks
sigma_Sd, tau_Sd, sigma_Rd, tau_Rd = symbols(
    r"\sigma_{Sd}, \tau_{Sd}, \sigma_{Rd}, \tau_{Rd}",
)

_p = {
    sigma_Sd: 20 * u.MPa,
    tau_Sd: 15 * u.MPa,
    sigma_Rd: 250 * u.MPa,
    tau_Rd: 75 * u.MPa,
}

# Expressions to check
_expr = [
    sigma_Sd / sigma_Rd,
    tau_Sd / tau_Rd,
]

# Evaluate expressions
_v = {
    k: k | pc.subs(_e | _p) | pc.N for k in _expr
}

# Check if expressions are less than 1
_c = {
    k: check(v, 1.0) for k, v in _v.items()
}

# Specify float format only for the check values
_ff = {
    k: [None, "{:.3f}", None] for k in _c.keys()
}

show_eqn([_p | _v, _c], float_format=_ff)

<IPython.core.display.Latex object>

The type of comparison in the `check` function can be specified, as well as the value to be checked against:

In [21]:
from sympy import Eq, Ge, Gt, Le, Lt

a, b, c, d, f = symbols("a, b, c, d, f")

_expr = {
    a: (3, Le, 1),
    b: (4, Gt, 2),
    c: (5, Lt, 3),
    d: (6, Ge, 4),
    f: (7, Eq, 5),
}

_c = {
    k: check(lhs=v[0], test=v[1], rhs=v[2])
    for k, v in _expr.items()
}

show_eqn(
    [
        {k: v[0] for k, v in _expr.items()},
        _c,
    ],
)

<IPython.core.display.Latex object>

### Check Function Templates

The `check` function supports customizable templates for different visual styles:

In [22]:
# Default template behavior
result_default = [
    check(0.8, 1.0), # True
    check(1.2, 1.0), # False
]

# Using named template sets
result_boxed = [check(0.8, 1.0, template="boxed"), check(1.2, 1.0, template="boxed")]
result_minimal = [check(0.8, 1.0, template="minimal"), check(1.2, 1.0, template="minimal")]

# Demonstration of different template styles
template_demo = {
    "Default": result_default,
    "Boxed": result_boxed,
    "Minimal": result_minimal,
}

# template_demo is dict[Any, list], therefore it must converted to Dataframe before passing to show_eqn

from keecas import Dataframe

show_eqn(
    Dataframe(template_demo),
    col_wrap=[None, r":\quad", r"&\quad"],
    environment="align*",
)

<IPython.core.display.Latex object>

In [23]:
# Complex expression with multiple substitutions
a, b, c, d = symbols(r"a, b, c, d")

_p = {
    a: 3,
    b: 4,
}
params.update(_p)

_e = {
    d: "sqrt(a^2 + b^2) / c" | pc.parse_expr,  # Uses c, which is defined below
    c: "a*b" | pc.parse_expr,                  # Defined after d, but keecas handles it
}
eqn.update(_e)

_v = {
    k: v | pc.subs(eqn | params) | pc.N
    for k, v in _e.items()
}

show_eqn([_p | _e, _v], float_format="{:.3f}")

<IPython.core.display.Latex object>

### Custom Templates

You can also use completely custom templates:

In [24]:
# Custom template examples (emoji may not render in latex)
custom_success = r"✅ ${symbol}{rhs}$ \textbf{{PASS}}"
custom_failure = r"❌ ${symbol}{rhs}$ \textbf{{FAIL}}"

# Test both success and failure cases
ratio_ok = 0.8      # Should pass
ratio_fail = 1.2    # Should fail

check_ok = check(
    ratio_ok,
    1.0,
    success_template=custom_success,
    failure_template=custom_failure,
)

check_fail = check(
    ratio_fail,
    1.0,
    success_template=custom_success,
    failure_template=custom_failure,
)

custom_demo = {
    "Pass": check_ok,
    "Fail": check_fail,
}

show_eqn(custom_demo, col_wrap=[None, ":", "&"])

<IPython.core.display.Latex object>

### Template Configuration

Templates can also be configured globally via TOML config files:

```toml
[check_templates.template_sets.minimal]
success = '${symbol}{rhs} \,\textcolor{{green}}{{\checkmark}}$'
failure = '${symbol}{rhs} \,\textcolor{{red}}{{\times}}$'
```

This allows you to set project-wide or user-wide styling for all check functions.

## Automatic Dependency Ordering

Using `pc.subs` will automatically order the substitutions pairs topologically, so you don't have to order them yourself (default ``sympy`` behavior). For example, let's define some mappings in any order:

In [25]:
any_order_params = {
    x: y**2, # it used `y` which is defined later
    y: a + 1, # `y` is a dependent value
    a: 3,
}

show_eqn(any_order_params, environment="cases")

<IPython.core.display.Latex object>

If we use the default behavior of `sympy`'s `subs` method (unsorted substitutions) we get:

In [26]:
# `pc.subs(any_order_params, sorted=False)` is equivalent to `sympy`'s subs method
_v = {
    lhs: rhs | pc.subs(any_order_params, sorted=False)
    for lhs, rhs in any_order_params.items()
}

_d = {
    x: "partially evaluated",
}

show_eqn(
    [_v, _d],
    environment="cases",
)

<IPython.core.display.Latex object>

While the default behavior of `pc.subs` is to sort the substitutions:

In [27]:
_v = {
    lhs: rhs | pc.subs(any_order_params)
    for lhs, rhs in any_order_params.items()
}

_d = {
    x: "fully evaluated",
}

show_eqn(
    [_v, _d],
    environment="cases",
)

<IPython.core.display.Latex object>

::: {.callout-note}

`pc.subs` automatically converts objects to `sympy` expressions before substitution. In the example above, $a$ maps to the Python `int` value `3`. When piping `int` through `pc.subs`, it works seamlessly. If you used `sympy`'s `subs()` method directly, you'd get an error since `int` objects don't have a `subs()` method. You'd need to write `S(rhs).subs(...)` to explicitly convert first.

:::

## Common Pitfalls

::: {.callout-warning}

### Important Patterns to Know

Understanding these common pitfalls will save you debugging time and ensure correct calculations.

:::

### Pitfall 1: LaTeX vs Plain Symbol Notation

Symbols are compared by their string representation, so LaTeX and plain notation create different objects:

In [28]:
from IPython.display import display

# Two different symbols even though they render similarly
sigma_latex = symbols(r"\sigma_{Sd}")
sigma_plain = symbols("sigma_Sd")

display(sigma_latex)
display(sigma_plain)

# They are NOT equal
print(f"Are they equal? {sigma_latex == sigma_plain}")  # False

# Using LaTeX notation ensures proper rendering and avoids mismatches

\sigma_{Sd}

σ_Sd

Are they equal? False


### Pitfall 2: Using `vals` dict in `pc.subs`

While maintaining a global `vals` dict for evaluated results can be useful, passing it to `pc.subs` will overwrite expressions with already evaluated values, preventing recalculation when parameters change.

In [29]:
# Initial setup - create fresh global dicts for this example
params_pitfall = {}
eqn_pitfall = {}
vals_pitfall = {}

x, y, z = symbols(r"x, y, z")

_p = {x: 2}
params_pitfall.update(_p)

_e = {
    y: "x + 1" | pc.parse_expr,  # y depends on x
    z: "y^2" | pc.parse_expr,     # z depends on y (and indirectly on x)
}
eqn_pitfall.update(_e)

_v = {
    k: v | pc.subs(eqn_pitfall | params_pitfall) | pc.N
    for k, v in _e.items()
}
vals_pitfall.update(_v)

show_eqn([_p | _e, _v])

<IPython.core.display.Latex object>

Now update the parameter and see the problem:

In [30]:
# Update parameter and recalculate
_p = {x: 3}  # Changed value
params_pitfall.update(_p)

expr_to_recalc = [y, z]

# INCORRECT: passing vals overwrites expressions with old evaluated values
_v_incorrect = {
    k: eqn_pitfall[k] | pc.subs(eqn_pitfall | params_pitfall | vals_pitfall)
    for k in expr_to_recalc
}

# CORRECT: don't pass vals
_v_correct = {
    k: eqn_pitfall[k] | pc.subs(eqn_pitfall | params_pitfall)
    for k in expr_to_recalc
}

# Verify the difference using check()
_c = {
    k: check(_v_incorrect[k], _v_correct[k], Eq)
    for k in expr_to_recalc
}

show_eqn([_p | _v_incorrect, _c])

<IPython.core.display.Latex object>

::: {.callout-tip}

**Recommendation**: Only pass `vals` to `pc.subs` if you're certain no dependent expressions need updating.

:::

{{< pagebreak >}}

# Summary

This example demonstrated keecas capabilities for engineering calculations in Quarto documents.

### Features Covered

- **Symbolic math with units** - SymPy expressions with Pint units
- **Pipe commands** - Functional composition (`pc.parse_expr`, `pc.subs`, `pc.convert_to`, `pc.N`)
- **LaTeX environments** - align, cases, equation, and custom environments
- **Label generation** - Manual, automatic, and unique hash-based labels
- **Verification** - `check()` function with customizable templates
- **Dependency ordering** - Automatic topological sorting with `pc.subs`

### Key Patterns

```python
# Cell-local dicts (live in one cell)
_p = {F_d: 10*u.kN}                              # Parameters
_e = {sigma: "F_d/A_load" | pc.parse_expr}       # Expressions
_v = {k: v | pc.subs(_e|_p) | pc.N for k,v in _e.items()}  # Values

# Global persistence (across cells)
params.update(_p)
eqn.update(_e)
```

### Common Pitfalls

::: {.callout-warning}
1. Use LaTeX notation: `symbols(r"\sigma_{Sd}")` not `symbols("sigma_Sd")`
2. Avoid passing `vals` to `pc.subs` - it prevents recalculation
:::

# Links

- [pypi]("https://pypi.org/project/keecas/")
- [Repository]("https://github.com/kompre/keecas")
- [Documentation]("https://kompre.github.io/keecas")
- [Issues]("https://github.com/kompre/keecas/issues")